# Huấn luyện VAE bằng GPU Google Colab từ VS Code

Notebook này được thiết kế để chạy bằng extension Google Colab trong VS Code. Chọn **Select Kernel → Colab → Auto Connect**, cấp quyền Google rồi chạy lần lượt từ trên xuống. Mã nguồn được clone vào VM Colab và checkpoint được lưu trên Google Drive để không mất khi runtime ngắt.

> Trước khi chạy, hãy commit và push phiên bản code mới nhất lên GitHub.

## 1. Xác nhận runtime GPU

In [ ]:
import subprocess
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'Runtime chưa có GPU. Hãy ngắt kết nối, chọn Colab GPU runtime rồi kết nối lại.'
    )

subprocess.run(['nvidia-smi'], check=True)
gpu = torch.cuda.get_device_properties(0)
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GB')

## 2. Mount Drive và cấu hình nơi lưu lâu dài

Checkpoint được lưu trên Google Drive. Dataset mặc định nằm trên SSD của VM để GPU không phải chờ đọc JPEG qua Drive. Đặt `PERSIST_DATA_ON_DRIVE = True` nếu ưu tiên giữ dataset qua các lần khởi động runtime.

In [ ]:
import os
from pathlib import Path

USE_GOOGLE_DRIVE = True
PERSIST_DATA_ON_DRIVE = False
REPO_URL = 'https://github.com/lucaspham102/DL_ImgGenModel.git'
PROJECT_DIR = Path('/content/DL_ImgGenModel')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE_ROOT = Path('/content/drive/MyDrive/DL_ImgGenModel_storage')
else:
    STORAGE_ROOT = Path('/content/DL_ImgGenModel_storage')

CKPT_DIR = STORAGE_ROOT / 'checkpoints'
if PERSIST_DATA_ON_DRIVE:
    DATA_ROOT = STORAGE_ROOT / 'data'
    KAGGLE_CACHE = STORAGE_ROOT / 'kagglehub_cache'
else:
    DATA_ROOT = Path('/content/DL_ImgGenModel_runtime/data')
    KAGGLE_CACHE = Path('/content/kagglehub_cache')
for path in (DATA_ROOT, CKPT_DIR, KAGGLE_CACHE):
    path.mkdir(parents=True, exist_ok=True)

# kagglehub giữ CelebA trên Drive thay vì cache tạm của VM.
os.environ['KAGGLEHUB_CACHE'] = str(KAGGLE_CACHE)
print('Data:', DATA_ROOT)
print('Checkpoints:', CKPT_DIR)
print('Kaggle cache:', KAGGLE_CACHE)

## 3. Đồng bộ mã nguồn từ GitHub

In [ ]:
import os
import subprocess
import sys

if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'],
        check=True,
    )
elif PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
    raise RuntimeError(f'{PROJECT_DIR} đã tồn tại nhưng không phải Git repository.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('Commit đang chạy:')
subprocess.run(['git', '-C', str(PROJECT_DIR), 'log', '-1', '--oneline'], check=True)

## 4. Cài dependency và xác nhận CUDA

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')],
    check=True,
)

import torch
assert torch.cuda.is_available(), 'PyTorch không nhận CUDA sau khi cài dependency.'
print('CUDA sẵn sàng:', torch.cuda.get_device_name(0))

## 5. Cấu hình W&B

Để tránh nhập key mỗi lần, có thể tạo secret `WANDB_API_KEY` trong Colab. Đặt `BACKEND = 'none'` nếu chưa dùng W&B.

In [ ]:
BACKEND = 'wandb'
WANDB_PROJECT = 'DL_ImageGenModel'
WANDB_ENTITY = '24022367-vietnam-national-university-hanoi'

if BACKEND == 'wandb':
    import wandb
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get('WANDB_API_KEY')
    except Exception:
        pass
    wandb.login(key=api_key) if api_key else wandb.login()
    print('W&B đã đăng nhập.')

## 6. Chạy thử VAE với Fashion-MNIST

Cell này xác nhận toàn bộ luồng tải dữ liệu, GPU, loss và checkpoint trước khi train CelebA.

In [ ]:
sanity_command = [
    sys.executable, 'models/vae/train.py',
    '--dataset', 'fashion_mnist',
    '--data-root', str(DATA_ROOT),
    '--ckpt-dir', str(CKPT_DIR),
    '--subset', '2000',
    '--epochs', '2',
    '--batch-size', '64',
    '--seed', '42',
    '--kl-warmup-epochs', '2',
    '--backend', BACKEND,
    '--project', WANDB_PROJECT,
    '--entity', WANDB_ENTITY,
]
subprocess.run(sanity_command, cwd=PROJECT_DIR, env=os.environ.copy(), check=True)

## 7. Huấn luyện VAE chính thức

Checkpoint gần nhất được tự động dùng để resume. Đặt `DATASET = 'fashion_mnist'` nếu muốn huấn luyện toàn bộ Fashion-MNIST trước.

In [ ]:
DATASET = 'celeba'
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 5e-4
LATENT_DIM = 128
KL_WEIGHT = 1.0
KL_WARMUP_EPOCHS = 10
SEED = 42

train_command = [
    sys.executable, 'models/vae/train.py',
    '--dataset', DATASET,
    '--data-root', str(DATA_ROOT),
    '--ckpt-dir', str(CKPT_DIR),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--lr', str(LEARNING_RATE),
    '--latent-dim', str(LATENT_DIM),
    '--kl-weight', str(KL_WEIGHT),
    '--kl-warmup-epochs', str(KL_WARMUP_EPOCHS),
    '--seed', str(SEED),
    '--backend', BACKEND,
    '--project', WANDB_PROJECT,
    '--entity', WANDB_ENTITY,
]

last_checkpoint = CKPT_DIR / f'vae_{DATASET}_last.pt'
if last_checkpoint.is_file():
    train_command.extend(['--resume', str(last_checkpoint)])
    print('Resume từ:', last_checkpoint)
else:
    print('Bắt đầu run mới.')

subprocess.run(train_command, cwd=PROJECT_DIR, env=os.environ.copy(), check=True)

## 8. Kiểm tra checkpoint đã lưu

In [ ]:
checkpoints = sorted(CKPT_DIR.glob('vae_*.pt'))
if not checkpoints:
    raise FileNotFoundError(f'Không tìm thấy checkpoint trong {CKPT_DIR}')
for checkpoint in checkpoints:
    print(f'{checkpoint.name}: {checkpoint.stat().st_size / 1024**2:.1f} MB')